## Creating Input Target Pairs

<div class="alert alert-block alert-success">
Creating a data loader that fetches input-target pairs in sliding window approach
</div>

In [3]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

In [4]:
import tiktoken

In [6]:
tokenizer=tiktoken.get_encoding("gpt2")
encoded_text=tokenizer.encode(raw_text)

In [8]:
print(len(encoded_text))

5145


In [13]:
context_size=5
x=encoded_text[:context_size]
y=encoded_text[1:context_size+1]

print(f"{x}")
print(f"    {y}")

[40, 367, 2885, 1464, 1807]
    [367, 2885, 1464, 1807, 3619]


In [14]:
# A sample illustration of Input-Target Pairs 
for i in range(1,context_size+1):
    x=encoded_text[:i]
    y=encoded_text[i]
    
    print(f"{x} ----> {y}")

[40] ----> 367
[40, 367] ----> 2885
[40, 367, 2885] ----> 1464
[40, 367, 2885, 1464] ----> 1807
[40, 367, 2885, 1464, 1807] ----> 3619


In [16]:
# A sample illustration of Decoded Input-Target Pairs  
for i in range (1,context_size+1):
    x=encoded_text[:i]
    y=encoded_text[i]
    
    print(f"{tokenizer.decode(x)} ----> {tokenizer.decode([y])}")

I ---->  H
I H ----> AD
I HAD ---->  always
I HAD always ---->  thought
I HAD always thought ---->  Jack


### Creating a dataset and dataset loader using PyTorch

In [18]:
!pip install torch


[notice] A new release of pip is available: 23.3.2 -> 25.1
[notice] To update, run: C:\Users\joelb\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [19]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self,text,tokenizer,max_length,stride):
        self.input_ids=[]
        self.target_ids=[]
        
        #Tokenize the entire text
        token_ids = tokenizer.encode(text,allowed_special={"<|endoftext|>"})
        
        #Using sliding window to chunk the book into overlapping sequence of max_length
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk=token_ids[i:i+max_length]
            target_chunk=token_ids[i+1:i+1+max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(output_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]